<a href="https://colab.research.google.com/github/kalawinka/policorp/blob/main/examples/policorp_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Code Politics: Practical Python workflows for analysing parliamentary debates with PoliCorp**

### **PoliCorp**

* an open resource for easy access to and analysis of processed political text data
* enables political scientists and multidisciplinary researchers to access parliamentary speeches

### Requirements

- Google account
-  recommended browser: Google Chrome

### Workshop introduction
Parliamentary debates are rich but complex sources: they contain speeches, interruptions, procedural remarks, speaker metadata, party affiliations, and policy topics. In this workshop, we turn a structured PoliCorp JSON export into an analysis-ready pandas DataFrame and use it to explore political communication at several levels.

We begin with data inspection and descriptive statistics, then create reusable indicators for speech length and interjections. After visualising the main patterns, we move to text-focused methods: BERTopic for unsupervised topic discovery, word clouds for quick vocabulary exploration, embedding-based semantic search, and sentiment classification of interjections.

### Links to Be Used in the Workshop



*   Google Colab Notebook: [https://colab.research.google.com/drive/1IAl3udt0z5CneCUJta6X0_xqVPQg_Wol?usp=sharing](https://colab.research.google.com/drive/1IAl3udt0z5CneCUJta6X0_xqVPQg_Wol?usp=sharing)
*   PoliCorp: [https://policorp.pollux-fid.de/](https://policorp.pollux-fid.de/)


### Dataset

Two dates were selected:

* **26 January 2022** – Debate on mandatory COVID-19 vaccination
* **11 February 2025** – Final Bundestag debate before the federal election


### Learning objectives
By the end of the workshop, you will be able to:

1. Upload and load a PoliCorp JSON dataset in Google Colab.
2. Inspect a dataset's schema, data types, and missing values.
3. Create derived variables from nested parliamentary speech data.
4. Summarise and visualise speeches by party, topic, and speaker.
5. Build and interpret a BERTopic topic model.
6. Explore political language with word clouds and semantic similarity.
7. Apply a pretrained sentiment model to parliamentary interjections.

> **Workshop note:** Run the notebook from top to bottom. Some later sections reuse objects created earlier, especially `df`, `speech_text`, `embedding_model`, and `OUTPUT_DIR`.


In [ ]:
# Install the text-analysis libraries that may not be available in a fresh Colab runtime.
# The quiet flag keeps installation output manageable during the workshop.
!pip -q install bertopic

In [ ]:
# Standard-library imports
import json
import re
import shutil
from collections import Counter

# Data analysis and visualisation
import pandas as pd
import matplotlib.pyplot as plt

# Google Colab file upload/download helper
from google.colab import files

In [ ]:
# Open Colab's upload dialog and store uploaded files in a dictionary.
uploaded = files.upload()

# 1. Loading and structuring the dataset
## From nested PoliCorp JSON to an analysis-ready DataFrame

This section uploads the source file, reads its JSON structure, extracts the speech records, and flattens nested metadata into tabular columns with `pandas.json_normalize`.

In [ ]:
# Select the first uploaded file. For a workshop exercise, we assume one dataset is uploaded.
FILE = list(uploaded.keys())[0]

# Read the JSON file into a Python dictionary.
with open(FILE, "r", encoding="utf-8") as f:
    raw = json.load(f)

# Display the source disclaimer supplied with the dataset.
print("Disclaimer / source:", raw.get("disclaimer", "No disclaimer provided"))

# Extract the list of speech records and flatten nested fields into DataFrame columns.
records = raw["data"]
df = pd.json_normalize(records)

print(f"\nLoaded {len(df)} speech records and {df.shape[1]} columns.")



---

# 2. Exploring and enriching the data
## 2.1 Inspecting schema, data types, and missing values

Before analysing content, we check how the dataset is structured. This helps identify unexpected data types, incomplete variables, and fields that may require cleaning.

In [ ]:
# Show every column and the data type pandas inferred for it.
print("\n--- Columns and data types ---")
print(df.dtypes)

# Count missing values and display the ten columns with the most missing entries.
print("\n--- Missing values per column: top 10 ---")
print(df.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
# have a look at the structure of the text_raw field
df.text_raw.iloc[0]

## 2.2 Building a descriptive profile
### Dates, institutional roles, parties, topics, and speakers

These frequency tables provide a first overview of the corpus and help us assess its time coverage, political composition, and most common subject areas.


In [ ]:
print("\n--- Dates represented in the corpus ---")
print(df["date"].unique())

print("\n--- Number of records by parliamentary role ---")
print(df["role"].value_counts())

print("\n--- Number of records by party ---")
print(df["party"].value_counts())

print("\n--- Number of unique speakers ---")
print(df["speaker_id"].nunique())


## 2.3 Creating derived speech indicators
### Measuring speech length, interruptions, procedural interventions, and named entities

The `text_raw` column contains a list of typed segments. The helper functions below traverse those lists and convert nested content into numerical variables that are easier to aggregate and visualise.

In [ ]:
def count_words_speech(text_raw):
    """Count whitespace-separated words in segments labelled as a speech."""
    return sum(len(seg["text"].split()) for seg in text_raw if seg.get("type") == "speech")

def count_speech_segments(text_raw):
    """Count speech segments. This is a segment count, not a linguistic sentence count."""
    return sum(1 for seg in text_raw if seg.get("type") == "speech")


def count_interjections(text_raw):
    """Count the number of interjection segments attached to a speech record."""
    return sum(1 for seg in text_raw if seg.get("type") == "interjection")


def count_calls_to_order(text_raw):
    """Count procedural segments labelled as calls to order."""
    return sum(1 for seg in text_raw if seg.get("type") == "call to order")


def count_named_entities(text_raw):
    """Count all named-entity annotations stored across the nested segments."""
    return sum(len(seg["ner"]) for seg in text_raw if seg.get("ner"))

# Apply each helper function to every speech record and create new analysis columns.
df["word_count"] = df["text_raw"].apply(count_words_speech)
df["sentence_count"] = df["text_raw"].apply(count_speech_segments)
df["interjection_count"] = df["text_raw"].apply(count_interjections)
df["cto_count"] = df["text_raw"].apply(count_calls_to_order)
df["ner_count"] = df["text_raw"].apply(count_named_entities)

# Exclude "not known" party for the party-level panels (procedural/unassigned speakers)
parties_df = df[df["party"] != "not known"].copy()

# ---------------------------------------------------------------
# Print basic descriptive stats
# ---------------------------------------------------------------

print("\n--- Speech length statistics in words ---")
print(df["word_count"].describe())

# ---------------------------------------------------------------
# Panel 1 data: Median and standard deviation of speech length for each party.
# ---------------------------------------------------------------
wc_stats = parties_df.groupby("party")["word_count"].agg(["median", "std"]).sort_values("median", ascending=False)
print("\n--- Median number of words per party ---")
print(wc_stats)

# ---------------------------------------------------------------
# Panel 2 data: Count speeches in each party-session combination, then summarise across sessions.
# ---------------------------------------------------------------
per_session_counts = parties_df.groupby(["party", "sessionno"]).size().unstack(fill_value=0)
session_stats = per_session_counts.agg(["median", "std"], axis=1).sort_values("median", ascending=False)
print("\n--- Median number of speeches per party per session --")
print(session_stats)

# ---------------------------------------------------------------
# Panel 3 data: top 10 topics
# ---------------------------------------------------------------
top_topics = df["topic"].value_counts().head(10)
print("\n--- Top 10 topics --")
print(top_topics)

# ---------------------------------------------------------------
# Panel 4 data: top 10 speakers by number of speeches
# ---------------------------------------------------------------
df_speaker = df[df["role"] != "presidency"]
top_speakers_count = df_speaker["name"].value_counts().head(10)
print("\n--- Top 10 speakers by number of speeches ---")
print(top_speakers_count)

# ---------------------------------------------------------------
# Panel 5 data: top 10 speakers by total word count
# ---------------------------------------------------------------
top_speakers_words = df_speaker.groupby("name")["word_count"].sum().sort_values(ascending=False).head(10)
print("\n--- Top 10 speakers by total word count ---")
print(top_speakers_words)

# ---------------------------------------------------------------
# Panel 6 data: top 10 Speakers by total interjection count
# ---------------------------------------------------------------
top_speakers_itj = df_speaker.groupby("name")["interjection_count"].sum().sort_values(ascending=False).head(10)
print("\n--- Top 10 interjected speakers ---")
print(top_speakers_itj)


## 2.4 Exporting the enriched dataset
### Saving reproducible intermediate results

Saving the DataFrame at this stage means that later analyses can begin from the enriched table without repeating all preprocessing steps.

In [ ]:
# Colab's writable working directory.
OUTPUT_DIR = "/content"

# Save a spreadsheet-friendly CSV file and download it.
csv_path = f"{OUTPUT_DIR}/dataset_analysis.csv"
df.to_csv(csv_path, index=False)
files.download(csv_path)

# Save records as UTF-8 JSON. `orient="records"` produces a list of row dictionaries.
json_path = f"{OUTPUT_DIR}/dataset_analysis.json"
df.to_json(json_path, orient="records", force_ascii=False)
files.download(json_path)


## 2.5 Visualising the exploratory results
### A six-panel overview of parties, topics, speakers, and interruptions

The dashboard combines several complementary views. Boxplots show distributions rather than only averages, while horizontal bar charts make ranked categories easy to compare.

In [ ]:
# ---------------------------------------------------------------
# Create a six-panel exploratory dashboard.
# ---------------------------------------------------------------
plt.style.use("seaborn-v0_8-whitegrid")
fig = plt.figure(figsize=(16, 14))
gs = fig.add_gridspec(3, 2, hspace=0.55, wspace=0.35)

colors = {
    "Bündnis 90/Die Grünen": "#4C9F70",
    "SPD": "#E3000F",
    "CDU/CSU": "#1C1C1C",
    "FDP": "#FFED00",
    "AfD": "#009EE0",
    "DIE LINKE": "#BE3075",
    "non-party": "#999999",
}

# Panel 1: distribution of speech length by party.
ax1 = fig.add_subplot(gs[0, 0])
order = wc_stats.index.tolist()
box_data = [parties_df.loc[parties_df["party"] == p, "word_count"].values for p in order]
bar_colors = [colors.get(p, "#4C72B0") for p in order]
bp = ax1.boxplot(box_data, tick_labels=order, patch_artist=True, showmeans=False, medianprops=dict(color="black"))
for patch, c in zip(bp["boxes"], bar_colors):
    patch.set_facecolor(c)
    patch.set_edgecolor("black")
    patch.set_linewidth(0.5)
ax1.set_title("Speech Length per Party\n(distribution of words/speech)", fontsize=11)
ax1.set_ylabel("Words per speech")
ax1.tick_params(axis="x", rotation=40)
for label in ax1.get_xticklabels():
    label.set_ha("right")

# Panel 2: distribution of speech counts across sessions for each party
ax2 = fig.add_subplot(gs[0, 1])
order2 = session_stats.index.tolist()
box_data2 = [per_session_counts.loc[p].values for p in order2]
bar_colors2 = [colors.get(p, "#4C72B0") for p in order2]
bp2 = ax2.boxplot(box_data2, tick_labels=order2, patch_artist=True, showmeans=False, medianprops=dict(color="black"))
for patch, c in zip(bp2["boxes"], bar_colors2):
    patch.set_facecolor(c)
    patch.set_edgecolor("black")
    patch.set_linewidth(0.5)
ax2.set_title("Speeches per Party per Session\n(distribution across sessions)", fontsize=11)
ax2.set_ylabel("Speeches per session")
ax2.tick_params(axis="x", rotation=40)
for label in ax2.get_xticklabels():
    label.set_ha("right")

# Panel 3: most frequent corpus topics.
ax3 = fig.add_subplot(gs[1, 0])
top_topics.sort_values().plot(kind="barh", ax=ax3, color="#8172B2", edgecolor="black", linewidth=0.5)
ax3.set_title("Top 10 Topics", fontsize=11)
ax3.set_xlabel("Number of speeches")

# Panel 4: speakers with the most speech records.
ax4 = fig.add_subplot(gs[1, 1])
top_speakers_count.sort_values().plot(kind="barh", ax=ax4, color="#55A868", edgecolor="black", linewidth=0.5)
ax4.set_title("Top 10 Speakers by Number of Speeches", fontsize=11)
ax4.set_xlabel("Number of speeches")

# Panel 5: speakers contributing the greatest number of words.
ax5 = fig.add_subplot(gs[2, 0])
top_speakers_words.sort_values().plot(kind="barh", ax=ax5, color="#C44E52", edgecolor="black", linewidth=0.5)
ax5.set_title("Top 10 Speakers by Total Word Count", fontsize=11)
ax5.set_xlabel("Total words")

# Panel 6: speakers whose records contain the most interjections.
ax6 = fig.add_subplot(gs[2, 1])
top_speakers_itj.sort_values().plot(kind="barh", ax=ax6, color="#DD8452", edgecolor="black", linewidth=0.5)
ax6.set_title("Top 10 Speakers by Total Interjection Count", fontsize=11)
ax6.set_xlabel("Number of interjections")

fig.suptitle("PoliCorp Bundestag Speeches — Exploratory Dashboard", fontsize=15, fontweight="bold", y=0.995)

#uncomment to save the figure in HTML format
#out_path = f"{OUTPUT_DIR}/pollux_combined_dashboard.png"
#plt.savefig(out_path, dpi=150, bbox_inches="tight")
#files.download(out_path)


---

# 3. Discovering themes with BERTopic

[https://maartengr.github.io/BERTopic/index.html](https://maartengr.github.io/BERTopic/index.html)
## 3.1 Importing the topic-modeling toolkit

BERTopic combines sentence embeddings, dimensionality reduction, clustering, and class-based TF-IDF to identify groups of semantically related documents and describe them with representative keywords.

In [ ]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer, util
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import CountVectorizer

## 3.2 Defining language-specific helpers
### Removing frequent German function words

A custom stop-word list reduces grammatical and parliamentary formulae that would otherwise dominate the topic labels. Domain-specific stop words should always be reviewed rather than treated as universally correct.

In [ ]:
# Define a custom list of common German stop words for BERTopic and wordcloud.
# Removing them helps BERTopic find more meaningful topic keywords.
GERMAN_STOPWORDS = [
    "der", "die", "das", "den", "dem", "des", "ein", "eine", "einer", "eines", "einem", "einen",
    "und", "oder", "aber", "auch", "als", "am", "an", "auf", "aus", "bei", "bis", "bin", "bist",
    "sind", "sein", "seine", "seiner", "seinem", "seinen", "sich", "so", "im", "in", "ist", "es",
    "ich", "sie", "wir", "er", "ihr", "ihre", "ihren", "ihrem", "ihrer", "mich", "mir", "uns",
    "man", "nach", "nicht", "noch", "nur", "ob", "schon", "sehr", "um", "von", "vor", "war",
    "waren", "was", "weil", "wenn", "wie", "wird", "werden", "wurde", "wurden", "zu", "zum",
    "zur", "über", "unter", "für", "mit", "durch", "dass", "dieser", "diese", "dieses", "diesem",
    "diesen", "dann", "doch", "hier", "haben", "hat", "hatte", "hatten", "habe", "kann", "können",
    "muss", "müssen", "soll", "sollen", "will", "wollen", "wollten", "gibt", "geht", "immer",
    "mehr", "sehr", "heute", "damit", "dabei", "dazu", "darauf", "davon", "dafür", "deshalb",
    "denn", "also", "wieder", "weiter", "alle", "allen", "aller", "alles", "andere", "anderen",
    "keine", "keinen", "kein", "einmal", "etwas", "gar", "ganz", "genau", "gerade", "jetzt",
    "natürlich", "nämlich", "eben", "erst", "einfach", "diesem", "welche", "welcher", "welchen",
    "worden", "werde", "würde", "würden", "können", "könnte", "könnten", "meine", "meiner",
    "meinen", "unser", "unsere", "unserer", "unseren", "euch", "euer", "eure", "sondern", "beim",
    "vom", "ins", "am", "vielen", "vielem", "dank", "herr", "frau", "kolleginnen", "kollegen",
    "kollege", "kollegin", "präsident", "präsidentin", "präsidium", "damen", "meine damen",
    "sehr geehrte", "geehrte", "geehrter", "liebe", "lieber",
    "ihnen", "ihm", "ihn", "herren", "mal", "sagen", "gesagt", "kommen", "kommt", "gut",
    "viel", "viele", "vielen", "vielleicht", "eigentlich", "richtig", "wollen", "sagt",
    "the", "und", "worden"
]

## 3.3 Preparing the document collection
### Combining speech segments and selecting substantive contributions

BERTopic expects one string per document. We therefore join all segments labelled `speech`, retain sufficiently long records, and remove presidency contributions that are often procedural.

In [ ]:
def combine_speech_segments(text_raw):
    """Join all segments labelled `speech` into one document string."""
    if not isinstance(text_raw, list):
        return ""

    return " ".join(
        item.get("text", "")
        for item in text_raw
        if item.get("type") == "speech"
    )


# Create one plain-text document for every row in the original DataFrame.
df["speech_text"] = [
    combine_speech_segments(value)
    for value in tqdm(df["text_raw"], total=len(df), desc="Combining speech segments")
]

# Use only longer speeches for better topic quality and exclude speeches of the president
topic_df = df[
    (df["word_count"] >= 30) &
    (df["role"] != "presidency")
].copy()

documents = topic_df["speech_text"].tolist()
print(f"Documents selected for topic modeling: {len(documents)}")



## 3.4 Configuring the BERTopic pipeline
### Embeddings, dimensionality reduction, clustering, and topic labels

Each component controls a different stage of the model. The settings below favour reproducibility and workshop-scale computation. KMeans is selected to create exactly 20 topics; the HDBSCAN alternative is retained for experimentation.


In [ ]:
# ------------------------------------------------
# 1. Embedding model
# ------------------------------------------------

# Create a SentenceTransformer model that converts each text/document into a numerical vector called an "embedding".
embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2", # Load the multilingual MiniLM model.
    device="cpu" # Run the embedding model on the CPU. Using "cuda" instead would run the model on a supported NVIDIA GPU.
)

# ------------------------------------------------
# 2. Better dimensionality reduction
# ------------------------------------------------

# Create a UMAP model to reduce high-dimensional vectors produced by MiniLM model to fewer dimensions while trying to preserve the semantic structure of the data.
# BERTopic uses the reduced vectors for clustering.
umap_model = UMAP(
    n_neighbors=30, # Number of neighboring documents UMAP considers when learning the structure of the embedding space.
    n_components=10, # Reduce each document embedding to 10 dimensions.
    min_dist=0.0,  # Controls how tightly UMAP is allowed to place points together. 0.0 allows very dense groups of documents.
    metric="cosine", # Use cosine distance when comparing sentence embeddings.
    random_state=42, # Fix the random seed for reproducibility
    low_memory=True #  Reduce memory usage during UMAP computation.
)

# ------------------------------------------------
# 3. Clustering
# ------------------------------------------------

# Create an HDBSCAN clustering model. HDBSCAN groups documents that are located close together in the UMAP-reduced embedding space.
hdbscan_model = HDBSCAN(
    min_cluster_size=10,  # Minimum number of documents required to form a cluster.
    min_samples=5,  # Controls how conservative HDBSCAN is when deciding whether a document belongs to a dense cluster. Higher values -> more documents may be classified as noise/outliers.
    metric="euclidean", # Use Euclidean distance for clustering.
    cluster_selection_method="eom", # Use the "Excess of Mass" cluster selection method.
    prediction_data=False  # Do not store additional information required for assigning completely new documents to existing clusters later -> saves memory.
)

# or

# Create a KMeans clustering model. KMeans divides all documents into a predefined number of clusters and each KMeans cluster becomes a topic.
kmeans_model = KMeans(
    n_clusters=20, # Force KMeans to create exactly 20 clusters.
    init="k-means++", # Controls how the initial cluster centers are selected. "k-means++" selects starting centers intelligently so that they are spread across the embedding space -> improves clustering compared with random centers.
    n_init=10, # Run KMeans initialization 10 times.
    max_iter=300, # Maximum number of optimization iterations for one KMeans run.
    random_state=42  # Fix the random seed for reproducibility
)

# ------------------------------------------------
# 4. German topic vocabulary
# ------------------------------------------------


# Create a CountVectorizer.
# The vectorizer builds the vocabulary that BERTopic uses to describe each discovered topic and is used later to extract topic words.
vectorizer_model = CountVectorizer(
    stop_words=GERMAN_STOPWORDS, # Remove the German stop words defined above before creating
    ngram_range=(1, 2), # Extract both single words and two-word phrases.
    min_df=2, # A word or phrase must occur in at least 2 documents. This reduces rare words, spelling errors, and vocabulary noise.
    max_df=0.90  # Ignore terms that appear in more than 90% of documents. Extremely frequent words usually do not help distinguish topics.
)

# ------------------------------------------------
# 5. Improve topic representation
# ------------------------------------------------

# Create a KeyBERT-inspired topic representation model -> improves the keywords used to describe each topic.
representation_model = KeyBERTInspired(
    top_n_words=10  # Keep the 10 most representative words or phrases for each
)

# ------------------------------------------------
# 6. BERTopic
# ------------------------------------------------


# Create the final BERTopic model.
# BERTopic combines all components defined above into one topic-modeling pipeline.
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    #hdbscan_model=hdbscan_model, # uncomment clustering mechanism that you want to use
    hdbscan_model=kmeans_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    calculate_probabilities=False,  # Do not calculate a probability distribution over all topics for every document -> saves memory
    verbose=True # Print progress information while BERTopic is running.
)

## 3.5 Fitting the topic model
### Assigning each selected speech to a discovered theme

`fit_transform` learns the topic structure and returns one topic identifier per document. We then attach those identifiers to the filtered DataFrame for later comparison with speaker and party metadata.

In [ ]:
# Fit all BERTopic components and assign a topic to every selected document.
topics, probabilities = topic_model.fit_transform(documents)

# Store topic assignments next to the corresponding speech metadata.
topic_df["bertopic_topic"] = topics

# Summarise discovered topics, document counts, and representative keywords.
topic_info = topic_model.get_topic_info()
topic_info.head()


---

## 3.6 Saving a trained BERTopic model
### Reusing a computationally expensive model

Model persistence stores the learned topic representation so it can be shared or reopened without fitting the complete pipeline again.

In [ ]:
import shutil

# 1. Save BERTopic model
MODEL_PATH = "/content/bundestag_bertopic_model"

topic_model.save(
    MODEL_PATH,
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 2. ZIP model folder
shutil.make_archive(
    "/content/bundestag_bertopic_model",
    "zip",
    MODEL_PATH
)

# 3. Download ZIP to your computer
files.download(
    "/content/bundestag_bertopic_model.zip"
)


## 3.7 Loading a saved BERTopic model
### Restoring a previously trained model in Colab

Upload the ZIP archive created in the previous section, unpack it, and reload the model with `BERTopic.load`.

In [ ]:
# Upload ZIP from your computer
uploaded = files.upload()

# Unzip
shutil.unpack_archive(
    "bundestag_bertopic_model.zip",
    "/content/bundestag_bertopic_model"
)

# Load BERTopic model
topic_model = BERTopic.load(
    "/content/bundestag_bertopic_model"
)

print("Model loaded!")



---



## 3.8 Inspecting topic assignments
### Connecting topic labels to individual speeches

Document-level information makes abstract topic clusters interpretable by showing each speech's assigned topic, representative terms, and related metadata.

In [ ]:
document_info = topic_model.get_document_info(documents)

document_info.head(15)

# Representative_document 0 False means BERTopic did not select it as one of the best examples of that topic.

## 3.9 Comparing topic keywords
### A bar-chart overview of the most important terms

This interactive figure displays the strongest words or phrases for the largest topics. Exporting it as HTML preserves Plotly's zoom and hover interactions.

In [ ]:
fig = topic_model.visualize_barchart(
    top_n_topics=20,
    n_words=10
)

# Make the entire figure larger
fig.update_layout(
    height=1800,
    width=1400,
    margin=dict(
        l=100,
        r=100,
        t=100,
        b=100
    )
)

fig.show()

#uncomment to save the figure in HTML format
#fig.write_html(f"{OUTPUT_DIR}/bertopic_barchart.html")
#files.download(f"{OUTPUT_DIR}/bertopic_barchart.html")

## 3.10 Mapping topics in semantic space
### Exploring proximity between discovered themes

BERTopic projects topic representations into two dimensions. Nearby points indicate topics with more similar vocabularies or semantic content, but the plot should be treated as an exploratory map rather than an exact distance measure.

In [ ]:
topic_model.visualize_topics()

## 3.11 Examining topic similarity
### A heatmap of relationships between topic representations

The heatmap provides a pairwise view of topic similarity and can reveal overlapping themes, potential subtopics, or clusters that may merit merging.

In [ ]:
topic_model.visualize_heatmap()

---

# 4. Exploring vocabulary with word clouds
## 4.1 Importing text-cleaning and visualisation tools

Word clouds offer a quick, qualitative view of frequent vocabulary. They are useful for exploration and communication, but frequency alone does not indicate importance, distinctiveness, or political position.

In [ ]:
import re
from wordcloud import WordCloud

## 4.2 Cleaning text and defining a reusable word-cloud function
### Tokenisation, stop-word removal, and visual defaults

The cleaning function keeps alphabetic German tokens of at least three characters. The second helper turns cleaned text into a consistently sized `WordCloud` object.

In [ ]:
# Match alphabetic German-language tokens containing at least three characters.
TOKEN_RE = re.compile(r"[A-Za-zÄÖÜäöüß]{3,}")

def clean_text(text: str) -> str:
    """Lowercase, keep only alphabetic tokens (len>=3), drop stopwords."""
    tokens = TOKEN_RE.findall(text.lower())
    tokens = [t for t in tokens if t not in GERMAN_STOPWORDS]
    return " ".join(tokens)


def make_wordcloud(text: str) -> WordCloud:
    """Generate a word cloud, using a fallback phrase when no tokens remain."""
    return WordCloud(
        width=500, height=350, background_color="white",
        colormap="viridis", max_words=80, collocations=False,
    ).generate(text if text.strip() else "keine daten")

## 4.3 Comparing vocabulary across prominent speakers
### Word clouds for the nine speakers contributing the most words

The analysis excludes presidency remarks and aggregates all remaining speech text by speaker. This provides a visual entry point for comparing recurring vocabulary.

In [ ]:
# ---------------------------------------------------------------
# Wordclouds per top speaker (exclude pure presidency role so the
# clouds reflect political/substantive content rather than procedure)
# ---------------------------------------------------------------
non_presidency = df[df["role"] != "presidency"]
top_speakers = (
    non_presidency.groupby("name")["word_count"].sum().sort_values(ascending=False).head(9).index.tolist()
)

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
for ax, speaker in zip(axes.flat, top_speakers):
    text = " ".join(non_presidency.loc[non_presidency["name"] == speaker, "speech_text"])
    wc = make_wordcloud(clean_text(text))
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(speaker, fontsize=12)
    ax.axis("off")

fig.suptitle("Wordclouds — Top 9 Speakers by Total Word Count (excl. presidency remarks)",
             fontsize=15, fontweight="bold")
plt.tight_layout()


## 4.4 Comparing vocabulary across parties
### Aggregating all speech text within each party

Party-level word clouds highlight frequent language in each group's contributions. Differences may reflect agenda, speaking opportunities, or corpus composition, so comparisons should be interpreted alongside document counts.

In [ ]:
# ---------------------------------------------------------------
# Wordclouds per party (exclude 'not known' / 'non-party')
# ---------------------------------------------------------------
parties = [p for p in df["party"].unique() if p not in ("not known", "non-party")]
parties = sorted(parties)

n = len(parties)
ncols = 4
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows))
axes = axes.flat

for ax, party in zip(axes, parties):
    text = " ".join(df.loc[df["party"] == party, "speech_text"])
    wc = make_wordcloud(clean_text(text))
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(party, fontsize=13)
    ax.axis("off")

# Hide any unused subplots
for ax in list(axes)[n:]:
    ax.axis("off")

fig.suptitle("Wordclouds by Party", fontsize=16, fontweight="bold")
plt.tight_layout()

---

# 5. Finding speeches by semantic similarity
## 5.1 Moving beyond exact keyword matching

Semantic search compares vector embeddings rather than literal words. A speech can therefore match a query even when it uses different wording. The similarity threshold controls the precision–recall trade-off and should be validated for the research question.


In [ ]:
# Hugging Face Dataset offers convenient batched transformations.
from datasets import Dataset

In [ ]:
def find_sim(embedding, query_embedding, threshold=0.60):
  """Return `True` when cosine similarity exceeds the selected threshold."""
  similarity = util.pytorch_cos_sim(embedding, query_embedding).item()
  return similarity > threshold

In [ ]:
# Remove presidency records so the search focuses on substantive political speeches.
df_speaker = df[df["role"] != "presidency"]
ds = Dataset.from_pandas(df_speaker)

# Encode speech texts in batches. Returning NumPy arrays keeps the Dataset serialisable.
encoded_ds = ds.map(
    lambda x: {"embeddings":  embedding_model.encode(
        x['speech_text'], convert_to_tensor=True
    )},
    batched=True,
    desc="Encoding speeches")

In [ ]:
# Define the concept to search for. Queries can contain keywords or a natural-language phrase.
# Alternative example: "Klimaneutrale Modernisierung"
key_phrase = "covid-19, pandemic, vaccination"
key_phrase_encoded = embedding_model.encode(key_phrase, convert_to_tensor=True)

In [ ]:
# Compare every speech embedding with the query embedding and collect matches.
matches = list()
for sa in tqdm(encoded_ds, desc="Searching for similar speeches"):
  embedding= sa.get('embeddings')
  speech_text = sa.get('speech_text')
  speech_id = sa.get('speech_id')
  if  find_sim(embedding, key_phrase_encoded):
    dct = {"speech_id" : speech_id,
           "speech_text" : speech_text}
    matches.append(dct)
sim_df = pd.DataFrame(matches)
print(f"Semantically similar speeches found: {len(sim_df)}")
sim_df

---

# 6. Analysing sentiment in parliamentary interjections
## 6.1 Research focus and analytical caution

This section extracts interjections associated with one speaker and applies a pretrained German sentiment classifier. Model labels describe linguistic sentiment, not necessarily political stance, civility, irony, or the speaker's intention. Results should be inspected manually and reported with these limitations.


In [ ]:
# Hugging Face pipelines wrap tokenisation, model inference, and label formatting.
from transformers import pipeline

In [ ]:
# Select speech records for one focal speaker with more than 5 interjections.
# Replace the name to study another speaker present in the corpus.
FOCAL_SPEAKER = "Olaf Scholz"
sel_df = df[
    (df["interjection_count"] >= 5) &
    (df["name"] == FOCAL_SPEAKER)
].copy()

print(f"Selected speech records for {FOCAL_SPEAKER}: {len(sel_df)}")

In [ ]:
# Extract every interjection from the nested `text_raw` list.
interjection_records = []
for _, row in sel_df.iterrows():
    interjections = [
        element.get("text", "")
        for element in row["text_raw"]
        if element.get("type") == "interjection"
    ]

    for interjection in interjections:
        interjection_records.append({
            "speech_id": row["speech_id"],
            "interjection": interjection,
        })

itj_df = pd.DataFrame(interjection_records)
itj_df["word_count"] = itj_df["interjection"].str.split().str.len()
itj_df.head()


In [ ]:
# Load a pretrained German sentiment classifier from Hugging Face.
MODEL_NAME = "oliverguhr/german-sentiment-bert"
MAX_CHARS = 512  # keep inputs short; BERT truncates at 512 tokens anyway
sentiment_pipe = pipeline("sentiment-analysis", model=MODEL_NAME, tokenizer=MODEL_NAME)

In [ ]:
# Very short interjections often provide too little context for meaningful classification.
itj_df = itj_df[itj_df["word_count"] >= 7].reset_index(drop=True)
texts = itj_df["interjection"].str.slice(0, MAX_CHARS).tolist()

print(f"Scoring {len(texts)} inrejections...")
results = []
batch_size = 32
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    results.extend(sentiment_pipe(batch, truncation=True))
    if i % (batch_size * 10) == 0:
        print(f"  {i}/{len(texts)}")

itj_df["sentiment_label"] = [r["label"] for r in results]
itj_df["sentiment_score"] = [r["score"] for r in results]

print("Sentiment analysis complete.")

In [ ]:
# Inspect individual predictions before aggregating them.
itj_df.head()

In [ ]:
# Summarise how many interjections received each sentiment label.
itj_df["sentiment_label"].value_counts()

# Thank you!